# Fase 4: Ingeniería de características

En esta fase se generan nuevas variables a partir del dataset preprocesado, con el objetivo de enriquecer la información disponible para los modelos de Machine Learning y mejorar su capacidad predictiva.

## Objetivo de la fase

El objetivo de esta fase es crear nuevas características útiles para la predicción de asistencia a citas médicas.

Se buscará transformar variables existentes en atributos con mayor valor analítico, especialmente variables temporales, grupos de edad e indicadores de riesgo.

In [23]:
import pandas as pd
import numpy as np

In [24]:
path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-clean.csv"

df = pd.read_csv(path)

print("Dataset limpio cargado correctamente")
print(df.shape)
df.head()

Dataset limpio cargado correctamente
(110526, 12)


,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No_show
0,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,0
1,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,0
2,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,0
3,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,0
4,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,0


## Conversión de fechas

Como el dataset fue guardado en CSV, las columnas de fecha deben convertirse nuevamente a formato datetime antes de crear variables temporales.

In [25]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'], errors='coerce')
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], errors='coerce')

df[['ScheduledDay', 'AppointmentDay']].dtypes

ScheduledDay      datetime64[us, UTC]
AppointmentDay    datetime64[us, UTC]
dtype: object

## Copia de trabajo

Se crea una copia del dataset limpio para aplicar la ingeniería de características sin modificar directamente el archivo base de preprocesamiento.

In [26]:
df_feat = df.copy()

print("Copia para feature engineering creada")
print(df_feat.shape)

Copia para feature engineering creada
(110526, 12)


In [27]:
df_feat[['ScheduledDay', 'AppointmentDay']].head()

,ScheduledDay,AppointmentDay
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00


## Variable: waiting_days

Se calcula la cantidad de días entre la fecha en que se agenda la cita y la fecha en que se realiza.

Esta variable es una de las más importantes del proyecto, ya que tiempos de espera más largos pueden aumentar la probabilidad de inasistencia.

In [28]:
df_feat['ScheduledDay'] = pd.to_datetime(df_feat['ScheduledDay'], errors='coerce')
df_feat['AppointmentDay'] = pd.to_datetime(df_feat['AppointmentDay'], errors='coerce')

df_feat[['ScheduledDay', 'AppointmentDay']].head()

,ScheduledDay,AppointmentDay
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00


In [29]:
df_feat['waiting_days'] = (
    df_feat['AppointmentDay'].dt.floor('D') - df_feat['ScheduledDay'].dt.floor('D')
).dt.days

df_feat[['ScheduledDay', 'AppointmentDay', 'waiting_days']].head()

,ScheduledDay,AppointmentDay,waiting_days
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,0
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,0
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,0
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,0
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,0


In [30]:
print("Mínimo waiting_days:", df_feat['waiting_days'].min())
print("Máximo waiting_days:", df_feat['waiting_days'].max())
print("Valores negativos en waiting_days:", (df_feat['waiting_days'] < 0).sum())

Mínimo waiting_days: -6
Máximo waiting_days: 179
Valores negativos en waiting_days: 5


## Tratamiento de waiting_days negativos

Si existen registros con tiempos de espera negativos, se consideran inconsistencias temporales del dataset.

En esta fase se conservarán, pero quedarán documentados para revisión.

In [31]:
df_feat[df_feat['waiting_days'] < 0][['ScheduledDay', 'AppointmentDay', 'waiting_days']].head()

,ScheduledDay,AppointmentDay,waiting_days
27033,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,-1
55226,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,-1
64175,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,-1
71533,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,-6
72362,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,-1


## Variables temporales derivadas

Se extrae el día de la semana de:
- la fecha de agendado;
- la fecha de cita.

Esto puede ayudar a detectar patrones de comportamiento según el día.

In [32]:
df_feat['scheduled_weekday'] = df_feat['ScheduledDay'].dt.dayofweek
df_feat['appointment_weekday'] = df_feat['AppointmentDay'].dt.dayofweek

df_feat[['ScheduledDay', 'scheduled_weekday', 'AppointmentDay', 'appointment_weekday']].head()

,ScheduledDay,scheduled_weekday,AppointmentDay,appointment_weekday
0,2016-04-29 18:38:08+00:00,4,2016-04-29 00:00:00+00:00,4
1,2016-04-29 16:08:27+00:00,4,2016-04-29 00:00:00+00:00,4
2,2016-04-29 16:19:04+00:00,4,2016-04-29 00:00:00+00:00,4
3,2016-04-29 17:29:31+00:00,4,2016-04-29 00:00:00+00:00,4
4,2016-04-29 16:07:23+00:00,4,2016-04-29 00:00:00+00:00,4


In [33]:
weekday_map = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}

print("Scheduled weekday únicos:", sorted(df_feat['scheduled_weekday'].unique()))
print("Appointment weekday únicos:", sorted(df_feat['appointment_weekday'].unique()))

Scheduled weekday únicos: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]
Appointment weekday únicos: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)]


## Variable: is_same_day

Indica si la cita fue agendada y atendida el mismo día.

Esta característica puede ser útil, ya que citas el mismo día pueden comportarse distinto respecto a las ausencias.

In [34]:
df_feat['is_same_day'] = (df_feat['waiting_days'] == 0).astype(int)

df_feat['is_same_day'].value_counts()

is_same_day
0    71964
1    38562
Name: count, dtype: int64

## Variable: age_group

Se agrupa la edad en categorías para capturar patrones más generales que podrían no observarse con la edad numérica sola.

In [35]:
def categorize_age(age):
    if age <= 12:
        return 'child'
    elif age <= 17:
        return 'teen'
    elif age <= 35:
        return 'young_adult'
    elif age <= 59:
        return 'adult'
    else:
        return 'senior'

df_feat['age_group'] = df_feat['Age'].apply(categorize_age)

df_feat[['Age', 'age_group']].head(10)

,Age,age_group
0,62,senior
1,56,adult
2,62,senior
3,8,child
4,56,adult
5,76,senior
6,23,young_adult
7,39,adult
8,21,young_adult
9,19,young_adult


In [36]:
df_feat['age_group'].value_counts()

age_group
adult          36350
young_adult    25624
senior         21173
child          21036
teen            6343
Name: count, dtype: int64

## Variable: has_comorbidity

Indica si el paciente presenta al menos una condición médica relevante:
- hipertensión
- diabetes
- alcoholismo
- discapacidad

In [37]:
df_feat['has_comorbidity'] = (
    (df_feat['Hipertension'] == 1) |
    (df_feat['Diabetes'] == 1) |
    (df_feat['Alcoholism'] == 1) |
    (df_feat['Handcap'] > 0)
).astype(int)

df_feat['has_comorbidity'].value_counts()

has_comorbidity
0    84114
1    26412
Name: count, dtype: int64

## Variable: risk_score

Se crea un puntaje simple de riesgo sumando factores que pueden influir en la asistencia:
- hipertensión
- diabetes
- alcoholismo
- discapacidad
- beca/apoyo social

Este puntaje no representa riesgo clínico formal, sino un indicador compuesto útil para modelado.

In [38]:
df_feat['risk_score'] = (
    df_feat['Hipertension'] +
    df_feat['Diabetes'] +
    df_feat['Alcoholism'] +
    (df_feat['Handcap'] > 0).astype(int) +
    df_feat['Scholarship']
)

df_feat['risk_score'].describe()

count    110526.000000
mean          0.418055
std           0.688467
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           4.000000
Name: risk_score, dtype: float64

## Variable: is_child_or_senior

Agrupa extremos etarios en una sola variable binaria:
- niños
- adultos mayores

Esto puede ayudar a detectar si ciertos grupos dependen más de acompañamiento o presentan patrones diferentes de asistencia.

In [39]:
df_feat['is_child_or_senior'] = ((df_feat['Age'] <= 12) | (df_feat['Age'] >= 60)).astype(int)

df_feat['is_child_or_senior'].value_counts()

is_child_or_senior
0    68317
1    42209
Name: count, dtype: int64

## Variable: appointment_month

Se extrae el mes de la cita para capturar posibles patrones temporales agregados.

In [40]:
df_feat['appointment_month'] = df_feat['AppointmentDay'].dt.month

df_feat['appointment_month'].value_counts().sort_index()

appointment_month
4     3235
5    80841
6    26450
Name: count, dtype: int64

## Variable: schedule_hour

Se extrae la hora en la que fue agendada la cita.

Esto puede aportar contexto operativo sobre el comportamiento de programación.

In [41]:
df_feat['schedule_hour'] = df_feat['ScheduledDay'].dt.hour

df_feat['schedule_hour'].describe()

count    110526.000000
mean         10.774542
std           3.216192
min           6.000000
25%           8.000000
50%          10.000000
75%          13.000000
max          21.000000
Name: schedule_hour, dtype: float64

## Revisión de nuevas variables

Se revisa la estructura del dataset enriquecido para validar que las nuevas características fueron creadas correctamente.

In [42]:
df_feat.columns.tolist()

['Gender',
 'ScheduledDay',
 'AppointmentDay',
 'Age',
 'Neighbourhood',
 'Scholarship',
 'Hipertension',
 'Diabetes',
 'Alcoholism',
 'Handcap',
 'SMS_received',
 'No_show',
 'waiting_days',
 'scheduled_weekday',
 'appointment_weekday',
 'is_same_day',
 'age_group',
 'has_comorbidity',
 'risk_score',
 'is_child_or_senior',
 'appointment_month',
 'schedule_hour']

df_feat.head()

In [43]:
new_features = [
    'waiting_days',
    'scheduled_weekday',
    'appointment_weekday',
    'is_same_day',
    'age_group',
    'has_comorbidity',
    'risk_score',
    'is_child_or_senior',
    'appointment_month',
    'schedule_hour'
]

df_feat[new_features].isnull().sum()

waiting_days           0
scheduled_weekday      0
appointment_weekday    0
is_same_day            0
age_group              0
has_comorbidity        0
risk_score             0
is_child_or_senior     0
appointment_month      0
schedule_hour          0
dtype: int64

## Resumen de variables creadas

Durante esta fase se generaron las siguientes características nuevas:

- `waiting_days`
- `scheduled_weekday`
- `appointment_weekday`
- `is_same_day`
- `age_group`
- `has_comorbidity`
- `risk_score`
- `is_child_or_senior`
- `appointment_month`
- `schedule_hour`

Estas variables enriquecen el dataset y aportan información temporal, demográfica y de riesgo que no estaba siendo aprovechada en la versión inicial del proyecto.

## Justificación técnica de la ingeniería de características

La versión inicial del proyecto utilizaba un conjunto muy limitado de variables, principalmente edad y condiciones médicas básicas. Sin embargo, la asistencia a citas médicas es un fenómeno influenciado también por factores temporales y contextuales.

Por esta razón, la ingeniería de características se centró en construir variables que capturen:
- el tiempo de espera entre agendado y cita;
- patrones por día de la semana;
- grupos etarios;
- acumulación de factores de riesgo;
- y características operativas del proceso de programación.

Esto permite ofrecer al modelo un conjunto de variables más expresivo y potencialmente más útil para mejorar la predicción de ausencias.

In [44]:
output_path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-features.csv"

df_feat.to_csv(output_path, index=False)

print("Dataset enriquecido guardado en:")
print(output_path)

Dataset enriquecido guardado en:
C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-features.csv


## Hallazgos clave de la ingeniería de características

A partir del dataset preprocesado se generaron nuevas variables que enriquecen significativamente la información disponible para el modelo.

Los principales hallazgos fueron:

- La variable `waiting_days` mostró tiempos de espera de entre -6 y 179 días.
- Se identificaron 5 registros con tiempos de espera negativos, lo que indica inconsistencias temporales en el dataset original.
- Las variables `scheduled_weekday` y `appointment_weekday` confirmaron que los registros se concentran entre lunes y sábado.
- La variable `age_group` permitió agrupar la edad en categorías más interpretables.
- La variable `risk_score` resumió factores médicos y sociales en un solo indicador compuesto.
- Todas las nuevas variables generadas quedaron sin valores nulos.

## Conclusión de la Fase 4

La ingeniería de características permitió transformar el dataset preprocesado en una versión más rica y útil para el entrenamiento de modelos de Machine Learning.

Las nuevas variables creadas aportan contexto temporal, segmentación por edad y señales compuestas de riesgo, lo cual representa una mejora importante respecto al enfoque inicial del proyecto.

Como resultado, se obtiene un dataset enriquecido y mejor preparado para comparar múltiples modelos en la siguiente fase.